In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)

from config.settings import *

import pandas as pd
import geopandas as gpd
import numpy as np
import fiona

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)

Project root: /Users/yongyili/EnvHealthLondon
Data dir: /Users/yongyili/EnvHealthLondon/data


In [2]:
os.listdir(DATA_DIR)

['laqn_borough.parquet',
 'London_Borough_Excluding_MHW.shp',
 'London_Borough_Excluding_MHW.shx',
 'laqn_raw.parquet',
 '.DS_Store',
 'greenspace_borough.parquet',
 'noise_borough.parquet',
 'opgrsp_gb.gpkg',
 'London_Borough_Excluding_MHW.dbf',
 'London_Borough_Excluding_MHW.prj',
 'imd_borough.parquet',
 'Road_Noise_Lden_England_Round_3.geojson',
 'laqn_clean.parquet',
 'imd_full.csv']

In [3]:
laqn_borough = pd.read_parquet(f"{DATA_DIR}/laqn_borough.parquet")
noise_borough = pd.read_parquet(f"{DATA_DIR}/noise_borough.parquet")
greenspace_borough = pd.read_parquet(f"{DATA_DIR}/greenspace_borough.parquet")
imd_borough = pd.read_parquet(f"{DATA_DIR}/imd_borough.parquet")

print("LAQN:", laqn_borough.shape)
print("Noise:", noise_borough.shape)
print("Greenspace:", greenspace_borough.shape)
print("IMD:", imd_borough.shape)

LAQN: (172, 5)
Noise: (33, 2)
Greenspace: (33, 2)
IMD: (33, 5)


In [4]:
print("LAQN columns:", laqn_borough.columns.tolist())
print("Noise columns:", noise_borough.columns.tolist())
print("Greenspace columns:", greenspace_borough.columns.tolist())
print("IMD columns:", imd_borough.columns.tolist())

LAQN columns: ['borough', 'year', 'no2_mean', 'no2_count', 'site_count']
Noise columns: ['borough_name', 'lden_mean']
Greenspace columns: ['borough_name', 'green_area_m2']
IMD columns: ['borough_name', 'imd_score_mean', 'imd_rank_mean', 'imd_decile_mean', 'pct_most_deprived']


In [5]:
no2_baseline = (
    laqn_borough[laqn_borough["year"].isin([2018, 2019])]
    .groupby("borough")["no2_mean"]
    .mean()
    .reset_index()
    .rename(columns={
        "borough": "borough_name",
        "no2_mean": "no2_baseline_mean"
    })
)

print(no2_baseline.shape)
no2_baseline.head()

(29, 2)


,borough_name,no2_baseline_mean
0,Barking and Dagenham,23.116494
1,Bexley,26.685649
2,Brent,46.512597
3,Bromley,30.871809
4,Camden,53.645008


In [6]:
print("NO2 boroughs:", no2_baseline["borough_name"].nunique())
print("Noise boroughs:", noise_borough["borough_name"].nunique())
print("Greenspace boroughs:", greenspace_borough["borough_name"].nunique())
print("IMD boroughs:", imd_borough["borough_name"].nunique())

NO2 boroughs: 29
Noise boroughs: 33
Greenspace boroughs: 33
IMD boroughs: 33


In [7]:
set(noise_borough["borough_name"]) - set(no2_baseline["borough_name"])

{'Barnet',
 'Hammersmith and Fulham',
 'Hounslow',
 'Kingston upon Thames',
 'Richmond upon Thames',
 'Waltham Forest'}

In [8]:
set(no2_baseline["borough_name"]) - set(noise_borough["borough_name"])

{'Kingston', 'Richmond'}

In [ ]:
# Fix name mismatch

In [9]:
no2_baseline["borough_name"] = no2_baseline["borough_name"].replace({
    "Kingston": "Kingston upon Thames",
    "Richmond": "Richmond upon Thames"
})

In [10]:
print(set(noise_borough["borough_name"]) - set(no2_baseline["borough_name"]))
print(set(no2_baseline["borough_name"]) - set(noise_borough["borough_name"]))

{'Barnet', 'Hammersmith and Fulham', 'Hounslow', 'Waltham Forest'}
set()


In [ ]:
# spatial nearest-station imputation for missing NO₂ boroughs
# “Because LAQN monitoring stations are unevenly distributed, missing borough-level NO₂ baseline values were imputed using nearest-neighbour borough interpolation based on borough centroid distance. These imputed values were flagged as estimated.”

In [11]:
boroughs = gpd.read_file(f"{DATA_DIR}/London_Borough_Excluding_MHW.shp")

boroughs = boroughs[["GSS_CODE", "NAME", "geometry"]].rename(
    columns={
        "GSS_CODE": "borough_code",
        "NAME": "borough_name"
    }
)

boroughs = boroughs.to_crs("EPSG:27700")

In [12]:
no2_full = boroughs[["borough_name", "geometry"]].merge(
    no2_baseline,
    on="borough_name",
    how="left"
)

no2_full["no2_imputed"] = no2_full["no2_baseline_mean"].isna()

print(no2_full[no2_full["no2_imputed"]][["borough_name"]])

              borough_name
3                 Hounslow
9                   Barnet
16          Waltham Forest
22  Hammersmith and Fulham


In [13]:
available = no2_full[~no2_full["no2_baseline_mean"].isna()].copy()
missing = no2_full[no2_full["no2_baseline_mean"].isna()].copy()

available["centroid"] = available.geometry.centroid
missing["centroid"] = missing.geometry.centroid

for idx, row in missing.iterrows():
    distances = available["centroid"].distance(row["centroid"])
    nearest_idx = distances.idxmin()
    nearest_borough = available.loc[nearest_idx, "borough_name"]
    nearest_value = available.loc[nearest_idx, "no2_baseline_mean"]
    
    no2_full.loc[idx, "no2_baseline_mean"] = nearest_value
    no2_full.loc[idx, "no2_source_borough"] = nearest_borough

no2_full["no2_source_borough"] = no2_full["no2_source_borough"].fillna(no2_full["borough_name"])

no2_baseline_full = no2_full[
    ["borough_name", "no2_baseline_mean", "no2_imputed", "no2_source_borough"]
].copy()

no2_baseline_full.head()

,borough_name,no2_baseline_mean,no2_imputed,no2_source_borough
0,Kingston upon Thames,44.309768,False,Kingston upon Thames
1,Croydon,39.499391,False,Croydon
2,Bromley,30.871809,False,Bromley
3,Hounslow,28.230027,True,Richmond upon Thames
4,Ealing,45.988016,False,Ealing


In [14]:
print(no2_baseline_full.shape)
print(no2_baseline_full.isnull().sum())

no2_baseline_full[no2_baseline_full["no2_imputed"]]

(33, 4)
borough_name          0
no2_baseline_mean     0
no2_imputed           0
no2_source_borough    0
dtype: int64


,borough_name,no2_baseline_mean,no2_imputed,no2_source_borough
3,Hounslow,28.230027,True,Richmond upon Thames
9,Barnet,46.512597,True,Brent
16,Waltham Forest,48.655825,True,Hackney
22,Hammersmith and Fulham,28.186041,True,Kensington and Chelsea


In [ ]:
# merge using the full NO₂ table

In [15]:
master = (
    no2_baseline_full
    .merge(noise_borough, on="borough_name", how="outer")
    .merge(greenspace_borough, on="borough_name", how="outer")
    .merge(imd_borough, on="borough_name", how="outer")
)

print(master.shape)
master.head()

(33, 10)


,borough_name,no2_baseline_mean,no2_imputed,no2_source_borough,lden_mean,green_area_m2,imd_score_mean,imd_rank_mean,imd_decile_mean,pct_most_deprived
0,Barking and Dagenham,23.116494,False,Barking and Dagenham,62.939922,6.590169e+06,32.883473,7244.500000,2.681818,3.636364
1,Barnet,46.512597,True,Brent,63.562162,1.748205e+07,15.953626,19287.450237,6.383886,0.473934
2,Bexley,26.685649,False,Bexley,63.082413,8.659319e+06,15.882363,19696.109589,6.506849,0.000000
3,Brent,46.512597,False,Brent,64.076461,6.242091e+06,25.199855,12034.705202,4.161850,5.780347
4,Bromley,30.871809,False,Bromley,63.344279,1.858615e+07,13.969447,21707.984772,7.106599,0.507614


In [16]:
master.isnull().sum()

borough_name          0
no2_baseline_mean     0
no2_imputed           0
no2_source_borough    0
lden_mean             0
green_area_m2         0
imd_score_mean        0
imd_rank_mean         0
imd_decile_mean       0
pct_most_deprived     0
dtype: int64